# Notebook 11 — Publication-grade figures and 96+ readiness validation (enhanced)

This version reads the fixed robustness tables and adds a few stronger interpretation layers:

- the original multi-year core figures
- **leadwise** AI-vs-HRES delta curves with confidence intervals
- distribution plots for long-lead Z500 deltas across windows
- regime-entropy robustness plots
- blocking-event skill plots if event metrics are available
- a more explicit validation table for publication readiness


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def detect_bundle_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        Path("/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass"),
    ]
    for base in candidates:
        if (base / "src" / "flagship_predictability").exists() and (base / "examples" / "default_settings.py").exists():
            return base
        alt = base / "flagship_predictability_nextpass"
        if (alt / "src" / "flagship_predictability").exists() and (alt / "examples" / "default_settings.py").exists():
            return alt
    raise RuntimeError("Could not detect bundle root; edit detect_bundle_root() with the correct project path.")

BUNDLE_ROOT = detect_bundle_root()
SRC_ROOT = BUNDLE_ROOT / "src"
NOTEBOOK_ROOT = BUNDLE_ROOT / "notebooks"
OUTPUT_ROOT = NOTEBOOK_ROOT / "outputs" / "flagship_96plus"
ROBUST_ROOT = OUTPUT_ROOT / "robustness_tables"
SELECTED_ROOT = OUTPUT_ROOT / "paper_96plus_selected"

for p in [BUNDLE_ROOT, SRC_ROOT]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

for sub in ["main_figures", "appendix_figures", "validation", "tables"]:
    (SELECTED_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Bundle root:", BUNDLE_ROOT)
print("Robustness root:", ROBUST_ROOT)
print("Selected root:", SELECTED_ROOT)


Bundle root: /global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass
Robustness root: /global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_96plus/robustness_tables
Selected root: /global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_96plus/paper_96plus_selected


In [2]:
def safe_read_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Could not read {path}: {e}")
        return pd.DataFrame()

def td_days(series):
    s = series.copy()
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce") / 24.0
    try:
        return pd.to_timedelta(s, errors="coerce").dt.total_seconds() / 86400.0
    except Exception:
        tmp = s.astype(str).str.extract(r"(-?\d+(?:\.\d+)?)")[0]
        return pd.to_numeric(tmp, errors="coerce") / 24.0

def set_pub_style():
    plt.rcParams.update({
        "figure.dpi": 170,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 13,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "lines.linewidth": 2.0,
        "lines.markersize": 5,
    })

set_pub_style()

coverage = safe_read_csv(ROBUST_ROOT / "window_coverage_summary.csv")
rank_stability = safe_read_csv(ROBUST_ROOT / "longlead_rank_stability.csv")
pairwise_summary = safe_read_csv(ROBUST_ROOT / "pairwise_longlead_deltas_summary.csv")
pairwise_by_window = safe_read_csv(ROBUST_ROOT / "pairwise_longlead_deltas_by_window.csv")
leadwise_pairwise_summary = safe_read_csv(ROBUST_ROOT / "pairwise_leadwise_deltas_summary.csv")
leadwise_pairwise_by_window = safe_read_csv(ROBUST_ROOT / "pairwise_leadwise_deltas_by_window.csv")
regime_balance = safe_read_csv(ROBUST_ROOT / "regime_balance_by_window.csv")
regime_entropy = safe_read_csv(ROBUST_ROOT / "regime_entropy_by_window.csv")
regime_order = safe_read_csv(ROBUST_ROOT / "regime_order_stability.csv")
blocking_penalty = safe_read_csv(ROBUST_ROOT / "blocking_penalty_by_window.csv")
blocking_summary = safe_read_csv(ROBUST_ROOT / "blocking_penalty_summary.csv")
blocking_event_skill = safe_read_csv(ROBUST_ROOT / "blocking_event_skill_summary.csv")
det_all = safe_read_csv(ROBUST_ROOT / "all_windows_deterministic_summary.csv")
dense_manifest = safe_read_csv(ROBUST_ROOT / "denselead_manifest_summary.csv")
winshare_by_lead = safe_read_csv(ROBUST_ROOT / "model_winshare_by_lead.csv")
integrity = safe_read_csv(ROBUST_ROOT / "robustness_integrity_summary.csv")

manifest_rows = []
def save_fig(fig, outdir: Path, stem: str, caption: str, tier: str):
    outdir.mkdir(parents=True, exist_ok=True)
    png = outdir / f"{stem}.png"
    pdf = outdir / f"{stem}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    manifest_rows.append({"tier": tier, "figure_stem": stem, "png_path": str(png), "pdf_path": str(pdf), "caption": caption})


In [3]:
# Figure 1 — all-years core skill
if not det_all.empty and {"tag", "variable", "model", "lead", "rmse_mean", "acc_mean"}.issubset(det_all.columns):
    sub = det_all[(det_all["tag"] == "all_2018_2022") & (det_all["variable"] == "z500")].copy()
    if not sub.empty:
        sub["lead_days"] = td_days(sub["lead"])
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), squeeze=False)
        ax = axes[0, 0]
        for model in sorted(sub["model"].unique()):
            ss = sub[sub["model"] == model].sort_values("lead_days")
            ax.plot(ss["lead_days"], ss["rmse_mean"], marker="o", label=model)
        ax.set_title("All-years Z500 RMSE")
        ax.set_xlabel("Lead (days)")
        ax.set_ylabel("Area-weighted RMSE")
        ax.legend(frameon=False)

        ax = axes[0, 1]
        for model in sorted(sub["model"].unique()):
            ss = sub[sub["model"] == model].sort_values("lead_days")
            ax.plot(ss["lead_days"], ss["acc_mean"], marker="o", label=model)
        ax.set_title("All-years Z500 ACC")
        ax.set_xlabel("Lead (days)")
        ax.set_ylabel("ACC")
        ax.set_ylim(max(0, sub["acc_mean"].min() - 0.05), 1.02)
        ax.legend(frameon=False)

        fig.suptitle("All-years deterministic core skill (2018–2022)", y=1.02)
        save_fig(fig, SELECTED_ROOT / "main_figures", "fig01_all_years_core_skill",
                 "All-years Z500 deterministic skill across the common lead ladder.", "main")


In [4]:
# Figure 2 — by-year long-lead deltas vs HRES
if not pairwise_by_window.empty:
    sub = pairwise_by_window[(pairwise_by_window["bucket"] == "year") & (pairwise_by_window["metric"] == "rmse_mean")].copy()
    years = [x for x in sorted(sub["tag"].astype(str).unique()) if str(x).isdigit()]
    vars_keep = [v for v in ["z500", "t850", "u850", "v850"] if v in set(sub["variable"])]
    pairs_keep = [p for p in ["GraphCast_minus_HRES", "NeuralGCM_minus_HRES"] if p in set(sub["pair"])]
    if years and vars_keep and pairs_keep:
        fig, axes = plt.subplots(1, len(pairs_keep), figsize=(5.3 * len(pairs_keep), 4.6), squeeze=False)
        for ax, pair in zip(axes.ravel(), pairs_keep):
            mat = sub[sub["pair"] == pair].pivot_table(index="tag", columns="variable", values="delta").reindex(index=years, columns=vars_keep)
            im = ax.imshow(mat.to_numpy(), aspect="auto")
            ax.set_title(pair.replace("_minus_", " − "))
            ax.set_xticks(range(len(vars_keep)))
            ax.set_xticklabels(vars_keep)
            ax.set_yticks(range(len(years)))
            ax.set_yticklabels(years)
            ax.set_xlabel("Variable")
            ax.set_ylabel("Year")
            cbar = fig.colorbar(im, ax=ax, shrink=0.86)
            cbar.set_label("Long-lead RMSE delta")
        fig.suptitle("By-year AI-vs-HRES long-lead RMSE deltas", y=1.03)
        save_fig(fig, SELECTED_ROOT / "main_figures", "fig02_yearly_rmse_delta_heatmap",
                 "Year-by-year long-lead RMSE deltas against HRES for the AI systems.", "main")


In [5]:
# Figure 3 — seasonal long-lead delta summary
if not pairwise_summary.empty:
    sub = pairwise_summary[(pairwise_summary["bucket"] == "season") & (pairwise_summary["metric"] == "rmse_mean")].copy()
    vars_keep = [v for v in ["z500", "t850", "u850", "v850"] if v in set(sub["variable"])]
    seasons = [s for s in ["DJF", "MAM", "JJA", "SON"] if s in set(sub["season"])]
    pair = "GraphCast_minus_HRES" if "GraphCast_minus_HRES" in set(sub["pair"]) else (sub["pair"].iloc[0] if len(sub) else None)
    if pair and vars_keep and seasons:
        psub = sub[sub["pair"] == pair]
        mat = psub.pivot_table(index="season", columns="variable", values="delta_mean").reindex(index=seasons, columns=vars_keep)
        fig, ax = plt.subplots(figsize=(6.8, 4.6))
        im = ax.imshow(mat.to_numpy(), aspect="auto")
        ax.set_title(f"Seasonal long-lead RMSE delta: {pair.replace('_minus_', ' − ')}")
        ax.set_xticks(range(len(vars_keep)))
        ax.set_xticklabels(vars_keep)
        ax.set_yticks(range(len(seasons)))
        ax.set_yticklabels(seasons)
        ax.set_xlabel("Variable")
        ax.set_ylabel("Season")
        cbar = fig.colorbar(im, ax=ax, shrink=0.86)
        cbar.set_label("Mean long-lead RMSE delta")
        save_fig(fig, SELECTED_ROOT / "main_figures", "fig03_seasonal_rmse_delta_heatmap",
                 "Seasonal mean long-lead RMSE deltas against HRES.", "main")


In [6]:
# Figure 4 — rank stability share
if not pairwise_summary.empty:
    sub = pairwise_summary[(pairwise_summary["metric"] == "rmse_mean") & (pairwise_summary["bucket"].isin(["year", "season"]))].copy()
    vars_keep = [v for v in ["z500", "t850", "u850", "v850"] if v in set(sub["variable"])]
    pairs_keep = [p for p in ["GraphCast_minus_HRES", "NeuralGCM_minus_HRES"] if p in set(sub["pair"])]
    if vars_keep and pairs_keep:
        fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.3), squeeze=False, sharey=True)
        for ax, bucket in zip(axes.ravel(), ["year", "season"]):
            ss = sub[sub["bucket"] == bucket].copy()
            x = np.arange(len(vars_keep))
            width = 0.35
            for i, pair in enumerate(pairs_keep):
                vals = []
                for v in vars_keep:
                    tmp = ss[(ss["variable"] == v) & (ss["pair"] == pair)]
                    vals.append(float(tmp["share_favorable"].iloc[0]) if len(tmp) else np.nan)
                ax.bar(x + (i - 0.5) * width, vals, width=width, label=pair.replace("_minus_", " − "))
            ax.set_xticks(x)
            ax.set_xticklabels(vars_keep)
            ax.set_ylim(0, 1.02)
            ax.set_title(f"Share favorable by {bucket}")
            ax.set_ylabel("Share of windows favoring AI")
            ax.legend(frameon=False)
        fig.suptitle("Cross-window stability of the AI advantage", y=1.03)
        save_fig(fig, SELECTED_ROOT / "main_figures", "fig04_rank_stability_share",
                 "Share of yearly and seasonal windows in which the AI systems outperform HRES at long lead.", "main")


In [7]:
# Figure 5 — regime balance
if not regime_balance.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2), squeeze=False)
    ax = axes[0, 0]
    yr = regime_balance[regime_balance["bucket"] == "year"]
    ss = yr.sort_values("tag")
    if not ss.empty:
        ax.plot(ss["tag"], ss["min_fraction"], marker="o", label="Min regime fraction")
        ax.plot(ss["tag"], ss["max_fraction"], marker="o", label="Max regime fraction")
        ax.set_title("By-year regime balance")
        ax.set_xlabel("Year")
        ax.set_ylabel("Fraction")
        ax.legend(frameon=False)
        ax.tick_params(axis="x", rotation=45)

    ax = axes[0, 1]
    sea = regime_balance[regime_balance["bucket"] == "season"]
    if not sea.empty:
        order = [s for s in ["DJF", "MAM", "JJA", "SON"] if s in set(sea["season"])]
        grp = sea.groupby("season", as_index=False).agg(
            mean_min_fraction=("min_fraction", "mean"),
            mean_max_fraction=("max_fraction", "mean"),
        )
        grp["season"] = pd.Categorical(grp["season"], categories=order, ordered=True)
        grp = grp.sort_values("season")
        ax.plot(grp["season"], grp["mean_min_fraction"], marker="o", label="Mean min fraction")
        ax.plot(grp["season"], grp["mean_max_fraction"], marker="o", label="Mean max fraction")
        ax.set_title("Seasonal average regime balance")
        ax.set_xlabel("Season")
        ax.set_ylabel("Fraction")
        ax.legend(frameon=False)

    fig.suptitle("Regime-balance robustness beyond the pilot window", y=1.03)
    save_fig(fig, SELECTED_ROOT / "main_figures", "fig05_regime_balance_robustness",
             "By-year and seasonal regime-balance diagnostics.", "main")


In [8]:
# Figure 6 — blocking penalty by season
if not blocking_summary.empty:
    sub = blocking_summary[blocking_summary["bucket"] == "season"].copy()
    seasons = [s for s in ["DJF", "MAM", "JJA", "SON"] if s in set(sub["season"])]
    models = sorted(sub["model"].astype(str).unique())
    if seasons and models:
        fig, ax = plt.subplots(figsize=(8.8, 4.5))
        x = np.arange(len(seasons))
        width = 0.8 / max(1, len(models))
        for i, model in enumerate(models):
            vals = []
            for s in seasons:
                tmp = sub[(sub["season"] == s) & (sub["model"] == model)]
                vals.append(float(tmp["penalty_mean"].iloc[0]) if len(tmp) else np.nan)
            ax.bar(x + (i - (len(models)-1)/2) * width, vals, width=width, label=model)
        ax.set_xticks(x)
        ax.set_xticklabels(seasons)
        ax.set_xlabel("Season")
        ax.set_ylabel("Mean blocked − unblocked RMSE")
        ax.set_title("Blocked-flow penalty by season")
        ax.legend(frameon=False, ncol=min(3, len(models)))
        save_fig(fig, SELECTED_ROOT / "main_figures", "fig06_blocking_penalty_by_season",
                 "Seasonal mean blocked-flow penalty for each deterministic model.", "main")


In [9]:
# Figure 7 — leadwise AI-vs-HRES delta curves (new, publication-useful)
if not leadwise_pairwise_summary.empty:
    sub = leadwise_pairwise_summary[(leadwise_pairwise_summary["bucket"] == "year") &
                                    (leadwise_pairwise_summary["metric"] == "rmse_mean") &
                                    (leadwise_pairwise_summary["pair"].isin(["GraphCast_minus_HRES", "NeuralGCM_minus_HRES"]))].copy()
    vars_keep = [v for v in ["z500", "t850", "u850", "v850"] if v in set(sub["variable"])]
    pairs_keep = [p for p in ["GraphCast_minus_HRES", "NeuralGCM_minus_HRES"] if p in set(sub["pair"])]
    if vars_keep and pairs_keep:
        fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0), squeeze=False, sharex=True)
        for ax, var in zip(axes.ravel(), vars_keep):
            ss = sub[sub["variable"] == var].sort_values("lead_h")
            for pair in pairs_keep:
                pp = ss[ss["pair"] == pair].sort_values("lead_h")
                if pp.empty:
                    continue
                x = pp["lead_h"] / 24.0
                ax.plot(x, pp["delta_mean"], marker="o", label=pair.replace("_minus_", " − "))
                ax.fill_between(x, pp["delta_ci_lo"], pp["delta_ci_hi"], alpha=0.15)
            ax.axhline(0.0, linestyle="--", linewidth=1.0)
            ax.set_title(var)
            ax.set_xlabel("Lead (days)")
            ax.set_ylabel("RMSE delta")
            ax.legend(frameon=False)
        fig.suptitle("Leadwise AI-vs-HRES RMSE advantage across yearly windows", y=1.02)
        save_fig(fig, SELECTED_ROOT / "main_figures", "fig07_leadwise_rmse_advantage",
                 "Leadwise RMSE deltas against HRES with bootstrap confidence intervals across yearly windows.", "main")


In [10]:
# Appendix A1 — dense-lead support if available
if not dense_manifest.empty and "selected_dense_leads_h" in dense_manifest.columns:
    dense_det = safe_read_csv(OUTPUT_ROOT / "denselead_upgrade" / "deterministic" / "deterministic_summary.csv")
    if not dense_det.empty and {"variable", "model", "lead", "rmse_mean"}.issubset(dense_det.columns):
        sub = dense_det[dense_det["variable"] == "z500"].copy()
        sub["lead_days"] = td_days(sub["lead"])
        fig, ax = plt.subplots(figsize=(7.8, 4.5))
        for model in sorted(sub["model"].unique()):
            ss = sub[sub["model"] == model].sort_values("lead_days")
            ax.plot(ss["lead_days"], ss["rmse_mean"], marker="o", label=model)
        ax.set_title("All-years dense-lead Z500 RMSE")
        ax.set_xlabel("Lead (days)")
        ax.set_ylabel("RMSE")
        ax.legend(frameon=False)
        save_fig(fig, SELECTED_ROOT / "appendix_figures", "figA1_denselead_z500_rmse",
                 "Dense-lead all-years Z500 RMSE if the common stores support it.", "appendix")


In [11]:
# Appendix A2 — distribution of long-lead Z500 deltas across windows
if not pairwise_by_window.empty:
    sub = pairwise_by_window[(pairwise_by_window["metric"] == "rmse_mean") &
                             (pairwise_by_window["variable"] == "z500") &
                             (pairwise_by_window["pair"].isin(["GraphCast_minus_HRES", "NeuralGCM_minus_HRES"]))].copy()
    if not sub.empty:
        buckets = [b for b in ["year", "season"] if b in set(sub["bucket"])]
        fig, axes = plt.subplots(1, len(buckets), figsize=(5.6 * max(1, len(buckets)), 4.4), squeeze=False, sharey=True)
        for ax, bucket in zip(axes.ravel(), buckets):
            ss = sub[sub["bucket"] == bucket]
            pairs = [p for p in ["GraphCast_minus_HRES", "NeuralGCM_minus_HRES"] if p in set(ss["pair"])]
            data = [ss[ss["pair"] == p]["delta"].dropna().to_numpy() for p in pairs]
            if data:
                ax.boxplot(data, labels=[p.replace("_minus_", " − ") for p in pairs], showmeans=True)
                ax.axhline(0.0, linestyle="--", linewidth=1.0)
                ax.set_title(f"Long-lead Z500 delta distribution ({bucket})")
                ax.set_ylabel("RMSE delta")
        save_fig(fig, SELECTED_ROOT / "appendix_figures", "figA2_z500_delta_distribution",
                 "Distribution of long-lead Z500 RMSE deltas against HRES across yearly and seasonal windows.", "appendix")


/tmp/ipykernel_1497929/673853467.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=[p.replace("_minus_", " − ") for p in pairs], showmeans=True)
/tmp/ipykernel_1497929/673853467.py:14: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=[p.replace("_minus_", " − ") for p in pairs], showmeans=True)


In [12]:
# Appendix A3 — regime entropy robustness
if not regime_entropy.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2), squeeze=False)
    yr = regime_entropy[regime_entropy["bucket"] == "year"].sort_values("tag")
    ax = axes[0, 0]
    if not yr.empty:
        ax.plot(yr["tag"], yr["normalized_entropy"], marker="o")
        ax.set_title("By-year normalized regime entropy")
        ax.set_xlabel("Year")
        ax.set_ylabel("Normalized entropy")
        ax.tick_params(axis="x", rotation=45)

    sea = regime_entropy[regime_entropy["bucket"] == "season"]
    ax = axes[0, 1]
    if not sea.empty:
        grp = sea.groupby("season", as_index=False)["normalized_entropy"].mean()
        order = [s for s in ["DJF", "MAM", "JJA", "SON"] if s in set(grp["season"])]
        grp["season"] = pd.Categorical(grp["season"], categories=order, ordered=True)
        grp = grp.sort_values("season")
        ax.plot(grp["season"], grp["normalized_entropy"], marker="o")
        ax.set_title("Seasonal mean normalized regime entropy")
        ax.set_xlabel("Season")
        ax.set_ylabel("Normalized entropy")

    fig.suptitle("Regime-distribution entropy beyond the pilot window", y=1.02)
    save_fig(fig, SELECTED_ROOT / "appendix_figures", "figA3_regime_entropy",
             "Regime-entropy diagnostics across years and seasons.", "appendix")


In [13]:
# Appendix A4 — blocking event skill if available
if not blocking_event_skill.empty and "csi_mean" in blocking_event_skill.columns:
    sub = blocking_event_skill[blocking_event_skill["bucket"] == "season"].copy()
    seasons = [s for s in ["DJF", "MAM", "JJA", "SON"] if s in set(sub["season"])]
    models = sorted(sub["model"].astype(str).unique())
    if seasons and models:
        fig, ax = plt.subplots(figsize=(8.8, 4.5))
        x = np.arange(len(seasons))
        width = 0.8 / max(1, len(models))
        for i, model in enumerate(models):
            vals = []
            for s in seasons:
                tmp = sub[(sub["season"] == s) & (sub["model"] == model)]
                vals.append(float(tmp["csi_mean"].iloc[0]) if len(tmp) else np.nan)
            ax.bar(x + (i - (len(models)-1)/2) * width, vals, width=width, label=model)
        ax.set_xticks(x)
        ax.set_xticklabels(seasons)
        ax.set_xlabel("Season")
        ax.set_ylabel("CSI")
        ax.set_title("Blocking event skill by season")
        ax.legend(frameon=False, ncol=min(3, len(models)))
        save_fig(fig, SELECTED_ROOT / "appendix_figures", "figA4_blocking_event_csi",
                 "Seasonal blocking-event critical success index by model.", "appendix")


In [14]:
# Validation summary for publication strength
rows = []

def record(name, status, criterion, evidence, action):
    rows.append({
        "test_name": name,
        "status": status,
        "criterion": criterion,
        "evidence": evidence,
        "recommended_action": action,
    })

# 1) eligible coverage
if coverage.empty:
    record("coverage", "FAIL", "Need multi-year and multi-season coverage.", "window_coverage_summary.csv missing or empty.", "Run Notebook 10.")
else:
    n_years = int(coverage.loc[coverage["bucket"] == "year", "n_windows"].sum()) if "bucket" in coverage.columns else 0
    seasons = set(coverage.loc[coverage["bucket"] == "season", "season"].astype(str)) if "season" in coverage.columns else set()
    status = "PASS" if n_years >= 5 and {"DJF","MAM","JJA","SON"}.issubset(seasons) else "WARN"
    record("coverage", status, "Need all years plus all four seasons represented.",
           f"n_year_windows={n_years}; seasons_present={sorted(seasons)}.", 
           "Broaden or fix window plan if any years/seasons are missing.")

# 2) all-years core figure exists
sub = det_all[(det_all.get("tag","") == "all_2018_2022")] if not det_all.empty and "tag" in det_all.columns else pd.DataFrame()
record("all_years_core_skill", "PASS" if not sub.empty else "FAIL",
       "Need pooled all-years deterministic evidence.", 
       f"all_2018_2022 rows={len(sub)}.", 
       "Ensure pooled all-years window ran and was collected.")

# 3) AI advantage stability
if pairwise_summary.empty:
    record("ai_advantage_stability", "FAIL", "AI systems should beat HRES in most windows.", "pairwise summary missing.", "Run Notebook 10.")
else:
    z500 = pairwise_summary[(pairwise_summary["metric"] == "rmse_mean") & (pairwise_summary["variable"] == "z500")]
    favorable_share = float(z500["share_favorable"].mean()) if not z500.empty else np.nan
    status = "PASS" if np.isfinite(favorable_share) and favorable_share >= 0.80 else ("WARN" if np.isfinite(favorable_share) and favorable_share >= 0.65 else "FAIL")
    record("ai_advantage_stability", status,
           "AI systems should beat HRES in most z500 windows.",
           f"mean_share_favorable={favorable_share}.",
           "Keep claims comparative if the favorable share is only moderate.")

# 4) leadwise consistency available
record("leadwise_consistency", "PASS" if not leadwise_pairwise_summary.empty else "WARN",
       "A stronger paper should show where the advantage appears across lead time.",
       f"leadwise_rows={len(leadwise_pairwise_summary)}.",
       "Use only long-lead claims if leadwise summaries are unavailable.")

# 5) regime balance
if regime_balance.empty:
    record("regime_balance", "FAIL", "Regime balance should remain acceptable across windows.", "regime balance missing.", "Run Notebook 10.")
else:
    minf = float(pd.to_numeric(regime_balance["min_fraction"], errors="coerce").min())
    status = "PASS" if minf >= 0.08 else ("WARN" if minf >= 0.05 else "FAIL")
    record("regime_balance", status,
           "No window should collapse into tiny regimes.",
           f"minimum_regime_fraction={minf}.",
           "Keep regime claims soft if some windows are very imbalanced.")

# 6) regime entropy
record("regime_entropy", "PASS" if not regime_entropy.empty else "WARN",
       "Entropy is a useful complement to min/max regime share.",
       f"entropy_rows={len(regime_entropy)}.",
       "Optional but useful for interpretation.")

# 7) blocking penalty
if blocking_summary.empty:
    record("blocking_penalty", "FAIL", "Blocked-flow fragility should be summarized across windows.", "blocking summary missing.", "Run Notebook 10.")
else:
    pos_share = float(pd.to_numeric(blocking_summary.get("share_positive_penalty"), errors="coerce").mean())
    status = "PASS" if np.isfinite(pos_share) and pos_share >= 0.60 else ("WARN" if np.isfinite(pos_share) else "FAIL")
    record("blocking_penalty", status,
           "Blocked cases should usually be harder than unblocked cases.",
           f"mean_positive_penalty_share={pos_share}.",
           "If penalties are weak, keep blocked-flow claims descriptive rather than sweeping.")

# 8) dense lead support
dense_ok = (not dense_manifest.empty) and ("selected_dense_leads_h" in dense_manifest.columns)
record("dense_lead_support", "PASS" if dense_ok else "WARN",
       "Dense lead support strengthens the case that the ranking is not an artifact of sparse checkpoints.",
       f"dense_manifest_rows={len(dense_manifest)}; columns={list(dense_manifest.columns)}",
       "Keep the dense-lead figure in the appendix if only sparse leads are available.")

validation = pd.DataFrame(rows)
validation.to_csv(SELECTED_ROOT / "validation" / "publication_readiness_summary.csv", index=False)

fig_manifest = pd.DataFrame(manifest_rows)
fig_manifest.to_csv(SELECTED_ROOT / "tables" / "figure_manifest.csv", index=False)

display(validation)
display(fig_manifest)


,test_name,status,criterion,evidence,recommended_action
0,coverage,PASS,Need all years plus all four seasons represented.,"n_year_windows=5; seasons_present=['DJF', 'JJA...",Broaden or fix window plan if any years/season...
1,all_years_core_skill,FAIL,Need pooled all-years deterministic evidence.,all_2018_2022 rows=0.,Ensure pooled all-years window ran and was col...
2,ai_advantage_stability,FAIL,AI systems should beat HRES in most z500 windows.,mean_share_favorable=0.32.,Keep claims comparative if the favorable share...
3,leadwise_consistency,PASS,A stronger paper should show where the advanta...,leadwise_rows=400.,Use only long-lead claims if leadwise summarie...
4,regime_balance,WARN,No window should collapse into tiny regimes.,minimum_regime_fraction=0.0788043478260869.,Keep regime claims soft if some windows are ve...
5,regime_entropy,PASS,Entropy is a useful complement to min/max regi...,entropy_rows=24.,Optional but useful for interpretation.
6,blocking_penalty,WARN,Blocked cases should usually be harder than un...,mean_positive_penalty_share=0.4066666666666668.,"If penalties are weak, keep blocked-flow claim..."
7,dense_lead_support,PASS,Dense lead support strengthens the case that t...,dense_manifest_rows=1; columns=['all_years_tag...,Keep the dense-lead figure in the appendix if ...


,tier,figure_stem,png_path,pdf_path,caption
0,main,fig02_yearly_rmse_delta_heatmap,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,Year-by-year long-lead RMSE deltas against HRE...
1,main,fig03_seasonal_rmse_delta_heatmap,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,Seasonal mean long-lead RMSE deltas against HRES.
2,main,fig04_rank_stability_share,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,Share of yearly and seasonal windows in which ...
3,main,fig05_regime_balance_robustness,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,By-year and seasonal regime-balance diagnostics.
4,main,fig06_blocking_penalty_by_season,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,Seasonal mean blocked-flow penalty for each de...
5,main,fig07_leadwise_rmse_advantage,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,Leadwise RMSE deltas against HRES with bootstr...
6,appendix,figA1_denselead_z500_rmse,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,Dense-lead all-years Z500 RMSE if the common s...
7,appendix,figA2_z500_delta_distribution,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,Distribution of long-lead Z500 RMSE deltas aga...
8,appendix,figA3_regime_entropy,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,Regime-entropy diagnostics across years and se...
